In [1]:
!pip install email-validator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [email-validator]


In [31]:
import re
import smtplib
from email.message import EmailMessage
from typing import TypedDict

from email_validator import validate_email, EmailNotValidError
from langgraph.graph import StateGraph, END

/Users/disha/opt/anaconda3/lib/python3.9/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [3]:
import sys
print(sys.executable)

/Users/disha/opt/anaconda3/bin/python


In [4]:
!{sys.executable} -m pip install email-validator

  Using cached email_validator-2.3.0-py3-none-any.whl.metadata (26 kB)
Using cached email_validator-2.3.0-py3-none-any.whl (35 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [email-validator]


In [4]:
from email_validator import validate_email, EmailNotValidError

In [6]:
from typing import TypedDict

In [7]:
#newsletter state
class NewsletterState(TypedDict):
    name: str
    email: str
    is_valid: bool
    status: str
    message: str

In [8]:
test_newsletter_state = {
    "name": "Disha",
    "email": "test@example.com",
    "is_valid": False,
    "status": "",
    "message": ""
}

test_newsletter_state

{'name': 'Disha',
 'email': 'test@example.com',
 'is_valid': False,
 'status': '',
 'message': ''}

In [18]:
# email validation tool 
from langchain_core.tools import tool

@tool
def validate_subscriber_email(email: str) -> str:
    """Validate whether an email address is properly formatted."""
    try:
        valid = validate_email(email, check_deliverability=False)
        return valid.email
    except EmailNotValidError as e:
        return f"Invalid email: {str(e)}"

In [19]:
# test
validate_subscriber_email.invoke({
    "email": "test@example.com"
})

'test@example.com'

In [20]:
# test
validate_subscriber_email.invoke({
    "email": "wrong-email"
})

'Invalid email: An email address must have an @-sign.'

In [22]:
# subscriber agent
def subscriber_agent(state: NewsletterState):
    name = state["name"]
    email = state["email"]

    validation_result = validate_subscriber_email.invoke({
        "email": email
    })

    if validation_result.startswith("Invalid email"):
        return {
            "is_valid": False,
            "status": "failed",
            "message": validation_result
        }

    return {
        "email": validation_result,
        "is_valid": True,
        "status": "validated",
        "message": f"Subscriber details validated for {name}."
    }

In [24]:
# test
subscriber_agent({
    "name": "Disha",
    "email": "test@example.com",
    "is_valid": False,
    "status": "",
    "message": ""
})

{'email': 'test@example.com',
 'is_valid': True,
 'status': 'validated',
 'message': 'Subscriber details validated for Disha.'}

In [35]:
SMTP_SERVER = "smtp-relay.brevo.com"
SMTP_PORT = 587

SMTP_LOGIN = "af92de001@smtp-brevo.com"
SMTP_PASSWORD = "xsmtpsib-999a8490dd2ccd169d2dbeea4e953de6a73fab5383295e3c4f8ae459118b7316-66ZIKxNH2BugjVRB"
SENDER_EMAIL = "your_verified_sender_email"  # the verified sender in Brevo
SENDER_NAME = "Newsletter Team"

In [26]:
from langchain_core.tools import tool
import smtplib
from email.message import EmailMessage

@tool
def send_confirmation_email(name: str, recipient_email: str) -> str:
    """Send newsletter subscription confirmation email."""

    try:

        msg = EmailMessage()

        msg["Subject"] = "Newsletter Subscription Confirmation"
        msg["From"] = SMTP_LOGIN
        msg["To"] = recipient_email

        msg.set_content(
f"""
Hello {name},

Thank you for subscribing to our newsletter.

Your subscription has been successfully confirmed.

Best regards,
Newsletter Team
"""
        )

        with smtplib.SMTP(SMTP_SERVER, SMTP_PORT) as server:

            server.starttls()

            server.login(
                SMTP_LOGIN,
                SMTP_PASSWORD
            )

            server.send_message(msg)

        return f"Email sent successfully to {recipient_email}"

    except Exception as e:
        return f"Email sending failed: {str(e)}"

In [27]:
send_confirmation_email.invoke({
    "name": "Disha",
    "recipient_email": "your_email@gmail.com"
})

'Email sent successfully to your_email@gmail.com'

In [28]:
# email agent 
def email_agent(state: NewsletterState):
    name = state["name"]
    email = state["email"]
    is_valid = state["is_valid"]

    if not is_valid:
        return {
            "status": "failed",
            "message": "Email was not sent because subscriber email is invalid."
        }

    email_result = send_confirmation_email.invoke({
        "name": name,
        "recipient_email": email
    })

    return {
        "status": "completed",
        "message": email_result
    }

In [29]:
email_agent({
    "name": "Disha",
    "email": "your_email@gmail.com",
    "is_valid": True,
    "status": "",
    "message": ""
})

{'status': 'completed',
 'message': 'Email sent successfully to your_email@gmail.com'}

In [32]:
# langgraph flow
newsletter_builder = StateGraph(NewsletterState)

newsletter_builder.add_node(
    "subscriber_agent",
    subscriber_agent
)

newsletter_builder.add_node(
    "email_agent",
    email_agent
)

newsletter_builder.set_entry_point(
    "subscriber_agent"
)

newsletter_builder.add_edge(
    "subscriber_agent",
    "email_agent"
)

newsletter_builder.add_edge(
    "email_agent",
    END
)

newsletter_graph = newsletter_builder.compile()

In [52]:
import requests
from langchain_core.tools import tool

EMAILJS_SERVICE_ID = "service_qip6hz9"
EMAILJS_TEMPLATE_ID = "template_pqz99n1"
EMAILJS_PUBLIC_KEY = "n-Q6pHKgQw9GAnylT"
EMAILJS_PRIVATE_KEY = "nGB6aDqUnKMfvZhDrOHYU"

In [53]:
@tool
def send_confirmation_email(name: str, recipient_email: str) -> str:
    """Send newsletter confirmation email using EmailJS."""

    url = "https://api.emailjs.com/api/v1.0/email/send"

    payload = {
        "service_id": EMAILJS_SERVICE_ID,
        "template_id": EMAILJS_TEMPLATE_ID,
        "user_id": EMAILJS_PUBLIC_KEY,
        "accessToken": EMAILJS_PRIVATE_KEY,
        "template_params": {
            "name": name,
            "email": recipient_email,
            "to_email": recipient_email,
            "to_name": name,
            "message": "Thank you for subscribing to our newsletter!"
        }
    }

    response = requests.post(url, json=payload)

    if response.status_code == 200:
        return f"Email sent successfully to {recipient_email}"
    else:
        return f"Email sending failed: {response.status_code} - {response.text}"

In [54]:
send_confirmation_email.invoke({
    "name": "Disha",
    "recipient_email": "dishapai1505@gmail.com"
})

'Email sent successfully to dishapai1505@gmail.com'